# Analysis of the first run of Chemeleon experiments

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt

In [13]:
df = pd.read_csv("results/chemeleon/all_runs.csv")

df["strategy"] = df.apply(
    lambda r: "frozen" if r.freeze_encoder
    else ("warmup" if r.head_only_epochs > 0 else "end-to-end"), axis=1)

reg = df[df.task == "regression"].copy()     
classif = df[df.task == "classification"].copy() 

print(f"Regression dataset length: {len(reg)} | Classification dataset length: {len(classif)}")
# print(pd.crosstab(df.task, df.strategy).to_string())

Regression dataset length: 17 | Classification dataset length: 24


17 runs for regression and 24 for classification

In [3]:
df.head(3)

,task,exp,seed,freeze_encoder,encoder_lr,head_lr,readout,hidden_dim,num_layers,dropout,...,weight_decay,grad_clip,head_only_epochs,reduce_lr,lr_factor,lr_patience,val_score,baseline_score,overrides,strategy
0,classification,manual,42,True,0.000010,0.001,mlp,512.0,3.0,0.1,...,0.0001,1.0,0,False,NaN,NaN,0.236693,0.0,model=chemeleon task=classification seed=42 de...,frozen
1,classification,manual,42,False,0.000010,0.001,mlp,512.0,3.0,0.1,...,0.0001,1.0,0,False,NaN,NaN,0.299808,0.0,model=chemeleon task=classification model.free...,end-to-end
2,classification,manual,42,False,0.000003,0.001,mlp,512.0,3.0,0.1,...,0.0001,1.0,0,False,NaN,NaN,0.264715,0.0,model=chemeleon task=classification model.free...,end-to-end


## Best configurations:

In [14]:
param_list = ['freeze_encoder', 'encoder_lr', 'head_lr', 'readout', 'hidden_dim', 
              'num_layers', 'dropout', 'epochs', 'batch_size', 'weight_decay',
                'head_only_epochs','reduce_lr', 'lr_factor', 'lr_patience', 'val_score', 
                'baseline_score', 'overrides', 'strategy']


best_reg = reg.loc[reg.val_score.idxmin()][param_list] # ST RAE smaller better
best_classif = classif.loc[classif.val_score.idxmax()][param_list] #MCC bigger better

In [15]:
print("Best regression run : ------------------------------------")
print(best_reg)

Best regression run : ------------------------------------
freeze_encoder                                                  False
encoder_lr                                                    0.00001
head_lr                                                         0.001
readout                                                           mlp
hidden_dim                                                      512.0
num_layers                                                        3.0
dropout                                                           0.1
epochs                                                             20
batch_size                                                         32
weight_decay                                                   0.0001
head_only_epochs                                                    0
reduce_lr                                                       False
lr_factor                                                         NaN
lr_patience                    

Best run is with end to end finetuning with encoder Lr = 0.00001, MLP with 512 hidden neurons readout with dropout and 20 epochs. 

Val score : 0.7 

Baseline score : 0.98 

In [16]:
print("Best classification run : ------------------------------------")
print(best_classif)

Best classification run : ------------------------------------
freeze_encoder                                                  False
encoder_lr                                                    0.00001
head_lr                                                         0.001
readout                                                           mlp
hidden_dim                                                      512.0
num_layers                                                        3.0
dropout                                                           0.1
epochs                                                             20
batch_size                                                         32
weight_decay                                                   0.0001
head_only_epochs                                                    0
reduce_lr                                                       False
lr_factor                                                         NaN
lr_patience                

Best classification run is for end to end finetuning with the same configuration as for regression task. 

Val score : 0.3

Baseline score : 0 

## Staregy analysis

In [17]:
for name, frame, lower in [("regression", reg, True),
                           ("classification", classif, False)]:
    g = (frame.groupby("strategy").val_score
         .agg(["count", "mean", "min", "max"]).round(4)
         .sort_values("mean", ascending=lower))
    print(f"\n{name}\n{g.to_string()}")


regression
            count    mean     min     max
strategy                                 
end-to-end     12  0.7140  0.7037  0.7246
warmup          3  0.7204  0.7109  0.7262
frozen          2  0.7360  0.7123  0.7598

classification
            count    mean     min     max
strategy                                 
warmup          5  0.2724  0.2607  0.2806
end-to-end     18  0.2712  0.2188  0.2998
frozen          1  0.2367  0.2367  0.2367


Hard to interpret - big imbalance in number of runs per strategy. But visible worse results for the frozen staregy in both tasks - end to end finetuning can have a positive impact.

In [18]:
# compare those strategies with all other params fixed:
BASE = ("readout == 'mlp' and hidden_dim == 512 and dropout == 0.1 and epochs == 20 "
        "and batch_size == 32 and weight_decay == 0.0001 and encoder_lr == 1e-5 "
        "and head_lr == 1e-3 and seed == 42 and reduce_lr == False")

for name, frame in [("regression", reg), ("classification", classif)]:
    print(f"\n{name}")
    print(frame.query(BASE)[["strategy", "head_only_epochs", "val_score", "exp"]]
          .to_string(index=False))


regression
  strategy  head_only_epochs  val_score                       exp
    frozen                 0   0.712252                    manual
end-to-end                 0   0.703750                    manual
end-to-end                 0   0.704678     task4/best_model_test
    warmup                 2   0.723975 task4/head_first_2_epochs
    warmup                 5   0.710947 task4/head_first_5_epochs

classification
  strategy  head_only_epochs  val_score                       exp
    frozen                 0   0.236693                    manual
end-to-end                 0   0.299808                    manual
end-to-end                 0   0.268877                    manual
end-to-end                 0   0.263578     task4/best_model_test
    warmup                 2   0.279560 task4/head_first_2_epochs
    warmup                 5   0.279826 task4/head_first_5_epochs


Here for resgression the difference between frozen and end to end is non-existent - end to end finetuning is not helping. However, for classification the frozen staregy has the lowest score and all other staregies are between 0.26-0.3